# Visa Buddy v2: evaluation on Colab (T4)

This notebook only *runs* the pipeline in `pipeline/`. It doesn't contain logic of its own, so it can't drift from the code that's scored and served.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**.
2. Accept the Llama 3.1 license on Hugging Face (`meta-llama/Llama-3.1-8B-Instruct`).
3. Add your token in Colab **Secrets** (key icon on the left) as `HF_TOKEN`, with notebook access on.

What this notebook runs: **step 4** (baseline, done) and **step 7** (the v2 fine-tune).

Step 4: answers the 66 frozen gold questions with **Llama 3.1 8B Instruct, no adapter**, using the same retriever and prompt the chat server uses, and grades each answer with the deterministic grader. Answers are saved one by one, so if Colab disconnects, run the notebook again and it resumes.

## 1. Get the code and pinned libraries

In [ ]:
import os
if not os.path.exists("/content/Visa_buddy"):
    !git clone https://github.com/Nichk2/Visa_buddy.git /content/Visa_buddy
%cd /content/Visa_buddy
!git pull --ff-only
!pip install -q -r requirements-gpu.txt

## 2. Check the GPU and the frozen gold set

In [ ]:
import torch, hashlib
assert torch.cuda.is_available(), "No GPU: switch the runtime to T4"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
expected = open("eval/gold_v1.sha256").read().split()[0]
actual = hashlib.sha256(open("eval/gold_v1.jsonl","rb").read()).hexdigest()
assert actual == expected, "gold_v1.jsonl changed since it was frozen"
print("gold_v1 sha256 OK:", actual[:12])

## 3. Hugging Face token

In [ ]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token loaded" if os.environ["HF_TOKEN"] else "missing HF_TOKEN secret")

## 4. Baseline: Llama 3.1 8B, no adapter

About 30–40 minutes on a T4. Safe to re-run after a disconnect; it skips answers already saved in `eval/baseline_predictions.jsonl`.

In [ ]:
!python -m pipeline.eval_answers --tag baseline

## 5. Results

In [ ]:
import json, pandas as pd
rep = json.load(open("eval/baseline_report.json"))
print("model:", rep["model"], "| quantization:", rep["quantization"], "| complete:", rep["complete"])
cols = ["n_answerable","citation_hit","fact_coverage","unsupported","false_refusal_rate","refusal_correct_on_must_refuse"]
table = pd.DataFrame({**rep["per_program"], "OVERALL": rep["overall"]}).T[cols]
table

In [ ]:
# A few answers to read by eye, including the must-refuse ones
preds = [json.loads(l) for l in open("eval/baseline_predictions.jsonl")]
for p in [p for p in preds if p["must_refuse"]][:3] + [p for p in preds if not p["must_refuse"]][:3]:
    print(f"--- {p['id']}  {p['grade']}\nQ: {p['question']}\nA: {p['answer'][:600]}\n")

## 6. Download the results

Unzip into your local `Visa_buddy/` folder, then run `python -m pipeline.merge_ledger` and `python -m pipeline.report` there. That keeps one ledger with every run, local and Colab.

In [ ]:
!cp runs/ledger.jsonl runs/ledger.colab.jsonl
!zip -q -r /content/visa_buddy_baseline_results.zip eval/baseline_predictions.jsonl eval/baseline_report.json runs/ledger.colab.jsonl
from google.colab import files
files.download("/content/visa_buddy_baseline_results.zip")

## 7. Step 7: the fine-tuned adapter (v2)

**Run sections 1, 2 and 3 first, then this section. Skip 4–6** (that's the baseline, already done).

Same 66 questions, same retriever, same prompt, same 4-bit loading as the baseline; the only change is the QLoRA adapter in `visa_buddy_data/models/v2/`. Resumable like the baseline. About 30–40 minutes on a T4.

In [ ]:
!git pull --ff-only
!python -m pipeline.eval_answers --tag v2 --adapter visa_buddy_data/models/v2 --gold eval/gold_v2.jsonl

In [ ]:
import json, pandas as pd
rep = json.load(open("eval/v2_report.json"))
print("adapter:", rep["adapter"], "| gold:", rep["gold_file"], "| complete:", rep["complete"])
cols = ["n_answerable","cite_when_answering","fact_coverage","unsupported","over_refusal_rate","refusal_correct_on_must_refuse"]
pd.DataFrame({**rep["per_program"], "OVERALL": rep["overall"]}).T[cols]

In [ ]:
from google.colab import files
!cp runs/ledger.jsonl runs/ledger.colab.jsonl
!zip -q -r /content/visa_buddy_v2_results.zip eval/v2_predictions.jsonl eval/v2_report.json runs/ledger.colab.jsonl
files.download("/content/visa_buddy_v2_results.zip")

## 8. Held-out safety set: baseline and v2 (before any fix)

**Run sections 1, 2 and 3 first, then this section. Skip 4–7.**

26 questions written after v2's rf08 failure and before any fix: 20 that must be refused (misrepresentation, evading the rules, legal strategy, personal predictions) and 6 honest versions of the same topics that should be answered. Both models, same prompt and retriever. About 10–15 minutes in total.

In [ ]:
!git pull --ff-only
!python -m pipeline.eval_answers --tag safety_baseline --gold eval/safety_v1.jsonl
!python -m pipeline.eval_answers --tag safety_v2 --adapter visa_buddy_data/models/v2 --gold eval/safety_v1.jsonl

In [ ]:
import json, pandas as pd
rows = {}
for tag in ["safety_baseline", "safety_v2"]:
    o = json.load(open(f"eval/{tag}_report.json"))["overall"]
    rows[tag] = {"must-refuse refused": o["refusal_correct_on_must_refuse"], "honest twins answered (not refused)": 1 - o["false_refusal_rate"],
                 "twins fact coverage": o["fact_coverage"]}
pd.DataFrame(rows).T

In [ ]:
from google.colab import files
!cp runs/ledger.jsonl runs/ledger.colab.jsonl
!zip -q -r /content/visa_buddy_safety_results.zip eval/safety_baseline_predictions.jsonl eval/safety_baseline_report.json eval/safety_v2_predictions.jsonl eval/safety_v2_report.json runs/ledger.colab.jsonl
files.download("/content/visa_buddy_safety_results.zip")